In [ ]:
%pip install -q openai

In [ ]:
# Load Data

import pandas as pd

df = pd.read_csv("./exp1.csv")

In [ ]:
# Set OpenAI API key

from openai import OpenAI

client = OpenAI(api_key = "")

In [ ]:
# All Currently Available Models via OpenAI API

models = client.models.list()

for model in sorted(models.data, key = lambda x: x.id):
    print(model.id)

In [ ]:
# Define Prompt Template

prompt_template = """
Using the 7-point scale below, rate the acceptability of the following sentence:

1. Strongly Unacceptable
2. Unacceptable
3. Somewhat Unacceptable
4. Neutral
5. Somewhat Acceptable
6. Acceptable
7. Strongly Acceptable

Sentence: "{}"

Just provide a numerical rating (1--7) for a given sentence.
"""

In [ ]:
# Test on Individual Sentences

sentence = "Martin is tough to encourage his students."

prompt = prompt_template.format(sentence)

response = client.chat.completions.create(
    model = "", # Model ID
    messages = [{"role": "user", "content": prompt}],
    max_completion_tokens = 1500
)

print(response.choices[0].message.content)
print(f"finish_reason: {response.choices[0].finish_reason}")
print(f"reasoning_tokens: {response.usage.completion_tokens_details.reasoning_tokens}")
print(f"completion_tokens: {response.usage.completion_tokens}")

In [ ]:
# Models

# gpt-5-2025-08-07
# gpt-5.1-2025-11-13
# gpt-5.2-2025-12-11
# gpt-5.4-2026-03-05
# gpt-5.5-2026-04-23

models = [
    "gpt-5-2025-08-07",
    "gpt-5.1-2025-11-13",
    "gpt-5.2-2025-12-11",
    "gpt-5.4-2026-03-05",
    "gpt-5.5-2026-04-23"
]

In [ ]:
# Prepare Combined Result Storage

combined_results = df.copy()

for model in models:
    combined_results[model] = ""

In [ ]:
# Run for Each Model and Populate the Respective Column

import time
from tqdm import tqdm

for model_name in models:
    print(f"Running model: {model_name}")

    for idx, row in tqdm(df.iterrows(), total = len(df), desc = f"Processing ({model_name})"):
        sentence = row["SENTENCE"]
        prompt = prompt_template.format(sentence)

        try:
            response = client.chat.completions.create(
                model = model_name,
                messages = [{"role": "user", "content": prompt}],
                max_completion_tokens = 1500
            )
            rating = response.choices[0].message.content.strip()
        except Exception as e:
            rating = f"Error: {str(e)}"

        combined_results.at[idx, model_name] = rating
        time.sleep(1.5)

In [ ]:
# Save Results

combined_results.to_csv("./exp1_gpt.csv", index = False)